In [ ]:
############################################################
#read in setup -- kind of stupid system actually
############################################################

import pandas as pd
from datetime import datetime, timedelta




with open('setup.py') as f:
    code = f.read()
exec(code)




############################################################
#EXECUTE in MODUELS/ FUNCTIONS -  MAKES SHARED NAMESPACE
############################################################

with open('functions.py') as f:
    code = f.read()
exec(code)





run_sql("use hoodalgo_db")






if 'driver' not in globals():
    driver = create_driver_profile_1()












#####################################
#Build Robinhood stock/etf  Universe
#####################################



#scrape_robinhood_universe(num_workers = 25)
df_stocks, df_etf = build_robinhood_universe(csv_path="robinhood_universe.csv")
stock_universe_list = df_stocks['symbol'].to_list()
etf_universe_list = df_etf['symbol'].to_list()









########################
#get a test ticker list 
########################





In [ ]:
#########################
#get test ticker list
#########################



stock_universe = scrape_wsj(driver = driver) #scrape wsj... just list to test for today. 
ticker_list_test = stock_universe['ticker'].to_list() #get a list of tickers 


#filter for [active] and [fractional-trading] based shares 
ticker_list_test = check_tickers_in_universe(ticker_list = ticker_list_test , universe_list = stock_universe_list )


len(ticker_list_test)

In [ ]:

##############################
#Scrapers -- all adds to db 
##############################





wsj_df = scrape_wsj(driver = driver)
trading_view_df = scrape_trading_view(driver = driver)
stock_twitts_trending = scrape_stocktwitts_trending(driver = driver)




df_crypto, crypto_list = scrape_robinhood_crypto()



✅ Deane’s MySQL Connector V39 — has query_value function 
Imported Selenium engine -- v14
📦 Switched default DB to: new_algo_db
functions read in 


NameError: name 'driver' is not defined

In [ ]:


################################
#FETCHERS COOKBOOK
################################




#Fetch Robinhoood API
#---------------------------------------------------------------------------------------------------------------------

robinhood_fundamentals_df = fetch_robinhood_fundamentals(ticker_list = stock_universe_list , chunk_size = 100)
robinhood_prices_df = fetch_robinhood_prices(ticker_list = stock_universe_list[:100] , chunk_size = 100)
robinhood_quotes_df = fetch_robinhood_quotes(ticker_list = stock_universe_list , chunk_size = 100)
robinhood_historicals = fetch_robinhood_historicals( ticker_list = stock_universe_list[0], interval = "5minute", span = "week") #dosent save... could but could be overkill. 
robinhood_options = fetch_robinhood_options(ticker_input = 'F') #not complete- not using so dont waste time on this

#---------------------------------------------------------------------------------------------------------------------










#Fetch Google News 
#---------------------------------------------------------------------------------------------------------------------


#api/rss version  -- Can get blocked 
google_news_df = run_fetch_google_news(ticker_list = ticker_list_test , period = '6h', workers = 20)




#Selenium Version -- less risk, slower
google_news_df = run_fetch_google_news_selenium(ticker_list = ticker_list_test[:10] , driver=driver, time_window="6h") #uses already created driver





#get argicle count works for both 
article_counts_df = get_news_counts(hours=6)


#---------------------------------------------------------------------------------------------------------------------












#Fetch stocktwits data 
#---------------------------------------------------------------------------------------------------------------------

#api version 
stocktwitts_messages_df = api_fetch_stocktwits_sentiment(ticker_list_test, max_workers = 10)
stock_twits_analysis_df = analize_stocktwits_sentiment(stocktwitts_messages_df)




#selenium version 
stocktwits_sentiment_df = selenium_fetch_stocktwits_sentiment(ticker_list = ticker_list_test, headless = False ) #has to be visual apparently 

#---------------------------------------------------------------------------------------------------------------------









In [ ]:
##########################################
#Loading pages and html injection blocks 
##########################################

#loading page 
driver.get("file://" + base_dir + "loading_page.html")
update_loading_page(driver, "Test", "Test")

In [ ]:
#######################
#Robinhood Functions
#######################


#Scrape Portfolio - opens new custom chrome every time 
current_overview, current_stocks, current_crypto = check_robinhood_portfolio(driver = driver)


market_buy_new(ticker = ticker_list_test[2], input_amount = 5, buy_in_type = 'Dollars', live = False, driver = driver)



In [ ]:
###############################
#Marginal Analysis Functions
################################



#marginal_analysis_stocktwitts_selenium = marginal_analysis_stocktwitts_selenium()


In [ ]:
#Filter Google News
google_news_df = run_sql("""

SELECT 
    ticker,
    COUNT(*) AS article_count,
    MAX(created_at) AS latest_article_time
FROM google_news_links
WHERE created_at >= NOW() - INTERVAL 224 HOUR
GROUP BY ticker
HAVING COUNT(*) > 2
ORDER BY article_count DESC;

""").to_df()

filtered_tickers = google_news_df['ticker'].to_list()


update_loading_page(driver, f"Complete - {len(filtered_tickers)} stocks passed filter ")


In [ ]:
driver.quit()

driver = create_driver_profile_1()

In [ ]:

df = selenium_fetch_stocktwits_sentiment(ticker_list = filtered_tickers , driver = driver) 

In [ ]:



    
    

    

#Filter by Quotes:  #during trading hours
################################################################################

hour = datetime.now().hour
after_hours = (hour < 9 or hour > 16)

if not after_hours:
    
    print("📊 running quote filter...")
    
    # scrape quotes
    quotes_df = fetch_robinhood_quotes(
        ticker_list = filtered_tickers,
        chunk_size = 100
    )
    
    print("before filter:", len(quotes_df))
    
    ########################################################
    # CLEAN (drop bad rows)
    ########################################################
    
    required_cols = ['spread','mid_price','bid_size','ask_size','symbol','pct_change']
    
    X = 0
    while X < len(required_cols):
        if required_cols[X] not in quotes_df.columns:
            print(f"⚠️ missing column: {required_cols[X]}")
        X += 1
    
    quotes_df = quotes_df.dropna(subset=[
        'spread','mid_price','bid_size','ask_size','pct_change'
    ])
    
    ########################################################
    # DERIVED METRICS
    ########################################################
    
    quotes_df['spread_pct'] = quotes_df['spread'] / quotes_df['mid_price']
    
    # order book imbalance
    quotes_df['imbalance'] = quotes_df['bid_size'] / (quotes_df['ask_size'] + 1)
    
    ########################################################
    # FILTER (REAL TRADABLE + MOVING)
    ########################################################
    
    quotes_df = quotes_df[
        (quotes_df['spread_pct'] <= 0.01) &          # tighter spreads (1%)
        (quotes_df['bid_size'] >= 20) &
        (quotes_df['ask_size'] >= 20) &
        (abs(quotes_df['pct_change']) >= 0.015)      # must be moving (1.5%+)
    ]
    
    print("after filter:", len(quotes_df))
    
    ########################################################
    # FALLBACK (if too aggressive)
    ########################################################
    
    if len(quotes_df) < 20:
    
        print("⚠️ too few tickers, relaxing filter...")
    
        quotes_df = quotes_df[
            (quotes_df['spread_pct'] <= 0.02) &      # relax to 2%
            (quotes_df['bid_size'] >= 10) &
            (quotes_df['ask_size'] >= 10) &
            (abs(quotes_df['pct_change']) >= 0.01)   # relax movement
        ]
    
        print("after relaxed filter:", len(quotes_df))
    
    ########################################################
    # SCORING (THIS IS THE EDGE)
    ########################################################
    
    quotes_df['score'] = (
        (1 / quotes_df['spread_pct']) * 0.5 +   # tighter = better
        quotes_df['imbalance'] * 2 +            # buying pressure
        abs(quotes_df['pct_change']) * 10       # movement
    )
    
    ########################################################
    # SORT BEST → WORST
    ########################################################
    
    quotes_df = quotes_df.sort_values(by='score', ascending=False)
    
    ########################################################
    # FINAL LIST
    ########################################################
    
    filtered_tickers = quotes_df['symbol'].tolist()
    
    print(f"✅ {len(filtered_tickers)} tickers passed quote filter")

################################################################################













    
    


#StockTwits Filter -- replace with booster not filter
##################################################################################################################
update_loading_page(driver, "Fetching Stocktwitts (API Version)")

#Scrape StockTwits
stocktwitts_messages_df = api_fetch_stocktwits_sentiment(filtered_tickers, max_workers = 10)
stock_twits_analysis_df = analize_stocktwits_sentiment(stocktwitts_messages_df)

#Filter StockTwits
stock_twits_analysis_df = stock_twits_analysis_df[

    # enough activity
    (stock_twits_analysis_df['message_volume'] >= 20) &

    # multiple people involved (not 1 guy spamming)
    (stock_twits_analysis_df['unique_users'] >= 15) &

    # real engagement
    (stock_twits_analysis_df['participation_ratio'] >= 0.3) &

    # directional bias
    (stock_twits_analysis_df['bullish_percent'] >= 60) &

    # strong overall signal
    (stock_twits_analysis_df['signal_score'] >= 150)
]

filtered_tickers = stock_twits_analysis_df['ticker'].tolist()


##################################################################################################################



#write filtered_tickers to csv (this is what starts the loop)
pd.DataFrame(filtered_tickers, columns=["ticker"]).to_csv("filtered_tickers.csv", index=False)



#Gal was to get 
update_loading_page(driver, f"Stock Universe Created {len(filtered_tickers)} tickers passed filter")
time.sleep(2)







In [ ]:
update_loading_page(driver, f"Stock Universe Created {len(filtered_tickers)} tickers passed filter")


In [ ]:
#######################
#Reposition Section
#######################


Y = 0 
while Y < len(filtered_tickers):

    attempt = 0
    
    while attempt < 3:
        try:
            robinhood_market_buy(
                ticker = filtered_tickers[Y],
                input_amount = 5,
                buy_in_type = 'Dollars',
                live = False,
                driver = driver
            )
            break
    
        except Exception as e:
            print(f"Attempt {attempt+1} failed:", e)
            time.sleep(0.3)
    
        attempt += 1
        
    time.sleep(.3)
    
    Y+=1


